In [0]:
%run ../00-common/config

In [0]:
from pyspark.sql import functions as F

orders = spark.table(f"{catalog_name}.{silver_schema}.orders")
reviews = spark.table(f"{catalog_name}.{silver_schema}.order_reviews")
customers = spark.table(f"{catalog_name}.{silver_schema}.customers")

customer_lookup = (
    customers
    .select(
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix"
    )
)

# nje review per porosi (mbaj me te fundit nese ka disa)
review_per_order = (reviews.select("order_id", "review_score")
    .groupBy("order_id").agg(F.max("review_score").alias("review_score")))

fact_orders = (orders
     .join(
        customer_lookup,
        on="customer_id",
        how="left"
    )
    .join(review_per_order, on="order_id", how="left")
    # date_key nga purchase (lidh me dim_date)
    .withColumn("date_key", F.date_format("order_purchase_timestamp", "yyyyMMdd").cast("int"))
    # on-time flag: dorezuar para ose ne daten e parashikuar
    .withColumn("is_on_time",
        F.when(F.col("order_delivered_customer_date").isNull(), None)
         .otherwise(F.col("order_delivered_customer_date") <= F.col("order_estimated_delivery_date")))
    .withColumn("is_delivered", F.col("order_status") == "delivered")
    .select(
        "order_id", "customer_id", "customer_unique_id", "customer_zip_code_prefix","date_key", "order_status",
        "review_score", "is_delivered", "is_on_time",
        "approval_hours", "handling_hours", "shipping_days",
        "total_delivery_days", "delivery_variance_days"
    ))

(fact_orders.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{gold_schema}.fact_orders"))
print(f"Wrote {fact_orders.count():,} orders")